In [23]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [24]:
from src.features.build_features import build_features

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error

from catboost import CatBoostRegressor

In [25]:
data = pd.read_parquet("../data/processed/move_ru_clean.parquet")

data = build_features(data)

data.head()

,deal_type,object_type,rooms,area_total,area_kitchen,area_living,price_per_m2,address,okrug,district,...,has_elevator,has_parking,has_renovation,kitchen_ratio,living_ratio,floor_ratio,is_first_floor,is_last_floor,house_age,is_new_building
0,sale,apartment,3.0,87.3,13.3,49.3,630011.45,"Москва, переулок Большой Симоновский, 2",ЦАО,пер Большой Симоновский,...,0.0,0.0,1.0,0.152348,0.564719,0.200000,0,0,11.0,0
1,sale,apartment,2.0,63.8,20.7,25.0,1057996.63,"Москва, проезд Кутузовский, 16",ЦАО,Дорогомилово,...,1.0,1.0,0.0,0.324451,0.391850,0.777778,0,0,NaN,1
2,sale,apartment,1.0,27.1,8.0,12.0,608856.09,"Москва, проспект 40 лет Октября",ЮВАО,Люблино,...,1.0,1.0,1.0,0.295203,0.442804,0.714286,0,0,NaN,1
3,sale,apartment,1.0,34.0,8.0,24.0,623529.41,"Москва, проспект 60-летия Октября, 12",ЮЗАО,пр-кт,...,0.0,0.0,1.0,0.235294,0.705882,0.500000,0,0,63.0,0
4,sale,apartment,3.0,112.0,32.0,80.0,1267857.14,"Москва, проспект Кутузовский",ЦАО,Хамовники,...,0.0,1.0,NaN,0.285714,0.714286,0.611111,0,0,NaN,1


In [26]:
target = "price_per_m2"

num_features = [
    "rooms",
    "area_total",
    "area_kitchen",
    "area_living",
    "metro_time_min",
    "floor",
    "total_floors",
    "year_built",
    "completion_year",
    "photo_count",
    "is_studio",
    "has_balcony",
    "has_elevator",
    "has_parking",
    "has_renovation",
    "kitchen_ratio",
    "living_ratio",
    "floor_ratio",
    "is_first_floor",
    "is_last_floor",
    "house_age",
    "is_new_building",
]

cat_features = [
    "okrug",
    "district",
    "metro",
    "house_type",
    "seller_type",
]

data[num_features] = data[num_features].fillna(-1)
data[cat_features] = data[cat_features].fillna("unknown")

X = data[num_features + cat_features]
y = data[target]

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cat_feature_index = [X.columns.get_loc(col) for col in cat_features]

In [28]:
model = CatBoostRegressor(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_feature_index,
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	learn: 178974.0217559	test: 174607.3248015	best: 174607.3248015 (0)	total: 10.2ms	remaining: 6.09s
100:	learn: 61011.3418992	test: 59606.6307717	best: 59606.6307717 (100)	total: 348ms	remaining: 1.72s
200:	learn: 49966.8055816	test: 48804.6369036	best: 48804.6369036 (200)	total: 720ms	remaining: 1.43s
300:	learn: 44441.8846579	test: 44049.5094519	best: 44049.5094519 (300)	total: 1.1s	remaining: 1.09s
400:	learn: 41034.9682310	test: 41441.6471577	best: 41441.6471577 (400)	total: 1.55s	remaining: 769ms
500:	learn: 38000.0751142	test: 38958.5331622	best: 38958.5331622 (500)	total: 2s	remaining: 395ms
599:	learn: 36009.2283911	test: 37388.9001576	best: 37388.9001576 (599)	total: 2.43s	remaining: 0us

bestTest = 37388.90016
bestIteration = 599



CatBoostRegressor(depth=6, eval_metric='MAE', iterations=600, learning_rate=0.05, loss_function='MAE', random_seed=42, verbose=100)

In [29]:
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
mape = mean_absolute_percentage_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("MAE:", round(mae, 2))
print("MAPE:", round(mape, 4))
print("R2:", round(r2, 4))

MAE: 37388.9
MAPE: 0.0749
R2: 0.9112


In [30]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.get_feature_importance()
}).sort_values("importance", ascending=False)

display(feature_importance.head(15))

,feature,importance
24,metro,25.770502
22,okrug,16.500036
23,district,8.287074
6,total_floors,7.834004
2,area_kitchen,5.252238
8,completion_year,4.312247
4,metro_time_min,3.813685
1,area_total,3.796875
26,seller_type,3.718088
21,is_new_building,2.659256


In [31]:
result = X_test.copy()
result["y_true"] = y_test.values
result["y_pred"] = preds
result["abs_error"] = (result["y_true"] - result["y_pred"]).abs()
result["ape"] = result["abs_error"] / result["y_true"]

result.sort_values("abs_error", ascending=False).head(10)

,rooms,area_total,area_kitchen,area_living,metro_time_min,floor,total_floors,year_built,completion_year,photo_count,...,is_new_building,okrug,district,metro,house_type,seller_type,y_true,y_pred,abs_error,ape
10303,2.0,107.50,32.30,33.40,6.0,4.0,14.0,-1.0,2025.0,1.0,...,1,ЦАО,Якиманка,Полянка,unknown,developer,2285488.37,7.360040e+05,1.549484e+06,0.677966
6134,3.0,140.35,54.37,53.07,15.0,13.0,16.0,-1.0,2028.0,1.0,...,1,ЦАО,Замоскворечье,Новокузнецкая,unknown,developer,3004288.65,1.830270e+06,1.174018e+06,0.390781
6110,4.0,169.97,54.42,80.45,15.0,12.0,16.0,-1.0,2028.0,1.0,...,1,ЦАО,Замоскворечье,Новокузнецкая,unknown,developer,2910316.49,1.750153e+06,1.160164e+06,0.398638
10398,2.0,87.20,24.30,39.00,10.0,21.0,27.0,-1.0,2028.0,1.0,...,1,ЦАО,Замоскворечье,Павелецкая,unknown,developer,2071146.79,1.123186e+06,9.479605e+05,0.457698
6109,4.0,169.97,54.42,80.45,15.0,10.0,16.0,-1.0,2028.0,1.0,...,1,ЦАО,Замоскворечье,Новокузнецкая,unknown,developer,2541103.18,1.702588e+06,8.385147e+05,0.329981
7713,2.0,138.90,52.37,41.45,10.0,6.0,6.0,-1.0,2024.0,1.0,...,1,ЦАО,Замоскворечье,Новокузнецкая,unknown,developer,2371719.94,1.696269e+06,6.754505e+05,0.284794
7740,1.0,57.70,24.37,16.95,10.0,3.0,6.0,-1.0,2024.0,1.0,...,1,ЦАО,Замоскворечье,Новокузнецкая,unknown,developer,1975733.10,1.394158e+06,5.815756e+05,0.294359
9486,2.0,67.30,14.70,22.10,8.0,2.0,15.0,-1.0,2027.0,1.0,...,1,ЮАО,ул Серпуховский Вал,Тульская,unknown,developer,1159300.00,6.046865e+05,5.546135e+05,0.478404
290,2.0,115.20,48.70,33.10,9.0,21.0,23.0,-1.0,2028.0,1.0,...,1,ЦАО,Замоскворечье,Новокузнецкая,unknown,unknown,2487000.00,1.957211e+06,5.297891e+05,0.213023
303,2.0,74.40,25.30,31.00,9.0,17.0,23.0,-1.0,2028.0,1.0,...,1,ЦАО,Замоскворечье,Новокузнецкая,unknown,unknown,2172000.00,1.644362e+06,5.276378e+05,0.242927


In [ ]:
os.makedirs("../models", exist_ok=True)

model.save_model("../models/catboost_price_per_m2.cbm")